# INT8 Post-Training_quantization (PTQ)
**Project:** Evaluating Neural Network Quantization for Efficient Deep Learning Inference

**Goal:** Quantize a pretrained ResNet-18 model to INT8 without additional training, then compare its predictive performance, serialized model size, and CPU inference latency against the FP32 baseline.

**Measurements:** Test accuracy, serialized model size, and CPU inference latency.

**Calculations:** Accuracy change, size reduction, and latency improvement, all compared to FP32 baseline.

**Dataset:** CIFAR-10

**Model:** Pretrained ResNet-18 fine-tuned for CIFAR-10 (SamAdamDay/resnet18_cifar10).

**Evaluation size:** The same fixed 1000 test images used in the FP32 baseline.


# Configuring the Colab notebook
Note: Select **CPU** for runtime if available

In [ ]:
import platform
import torch
import torchvision

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Using CPU") # Use CPU if available

Python: 3.13.16
PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
CUDA available: False
Using CPU


# Installing the module-loading library

In [ ]:
%pip install -q timm # Loading pretrained image-classification model from Hugging Face

import copy
import os
import random
import time
import json
import statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import timm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Setup complete.")

Setup complete.


# Checking TorchAO compatibility for quantization
Note: The session might require restarting after running the cell below

In [ ]:
%pip install -q torchao
%pip install --upgrade torchao
import torch
import torchao

print("PyTorch:", torch.__version__)
print("TorchAO:", torchao.__version__)

from torchao.quantization.pt2e.quantize_pt2e import (
    prepare_pt2e,
    convert_pt2e,
)

from torchao.quantization.pt2e.quantizer.x86_inductor_quantizer import (
    X86InductorQuantizer,
    get_default_x86_inductor_quantization_config,
)

print("PT2E imports successful!")

PyTorch: 2.11.0+cpu
TorchAO: 0.18.0
PT2E imports successful!


# Loading CIFAR-10 dataset for the ResNet-18
Note: The cell below might take approx. 25 minutes when first ran.

In [ ]:
# Loading CIFAR-10 dataset for the ResNet-18 model
from torch.utils.data import DataLoader, Subset

DATA_DIR = "./data"
BATCH_SIZE = 25
EVAL_SIZE = 1000

# Using the preprocessing specified by the model card
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.49139968, 0.48215827, 0.44653124),
        std=(0.24703233, 0.24348505, 0.26158768)
    )
])

train_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root=DATA_DIR,
    train=False,
    download=True,
    transform=transform
)

# Fixed and reproducible subset of the official test split
generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(
    len(test_dataset), generator=generator
)[:EVAL_SIZE].tolist()

test_subset = Subset(test_dataset, indices)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print("Training images available:", len(train_dataset))
print("Official test images:", len(test_dataset))
print("Images in this evaluation:", len(test_subset))


Training images available: 50000
Official test images: 10000
Images in this evaluation: 1000


# Loading the pre-trained ResNet-18 model

In [ ]:
MODEL_ID = "hf_hub:SamAdamDay/resnet18_cifar10"

model = timm.create_model(
    MODEL_ID,
    pretrained=True
)

model.eval()

print("Model loaded successfully.")
print("Architecture:", model.__class__.__name__)
print("Number of parameters:",
      sum(p.numel() for p in model.parameters()))
print("Number of output classes:", model.num_classes)

Model loaded successfully.
Architecture: ResNet
Number of parameters: 11181642
Number of output classes: 10


# Preparing the model for quantization
Note: The pretrained FP32 ResNet-18 model is copied to the CPU and prepared for static INT8 post-training-quantization (PTQ). The original model remains unchanged so that its predictions and performance can be compared with the quantized model.

In [ ]:
import copy

# Preserving the original FP32 model and create a CPU copy for INT8 PTQ
model_fp32_cpu = copy.deepcopy(model).cpu().eval()

# Creating a separate model to quantize
model_to_quantize = copy.deepcopy(model_fp32_cpu)

print("Model prepared for CPU quantization.")
print("Model device:", next(model_to_quantize.parameters()).device)

Model prepared for CPU quantization.
Model device: cpu


# Exporting and preparing the model

In [ ]:
# Use one real batch to define the model's input shape
from torch.utils.data import DataLoader, Subset

example_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=True,
)

example_images, _ = next(iter(example_loader))
example_inputs = (example_images.cpu(),)

# Export the model to a graph suitable for quantization
with torch.inference_mode():
    exported_model = torch.export.export(
        model_to_quantize,
        example_inputs,
    ).module()

# Configure the X86 CPU quantizer
quantizer = X86InductorQuantizer()
quantizer.set_global(
    get_default_x86_inductor_quantization_config()
)

# Insert observers and prepare the graph for calibration.
prepared_model = prepare_pt2e(exported_model, quantizer)

print("Model exported and prepared for PTQ.")

Model exported and prepared for PTQ.


/usr/local/lib/python3.13/dist-packages/torchao/quantization/pt2e/quantizer/x86_inductor_quantizer.py:1325: UserWarning: The input of maxpool2d is not quantized, skip annotate maxpool2d with config QuantizationConfig(input_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), output_activation=QuantizationSpec(dtype=torch.uint8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.HistogramObserver'>, eps=0.000244140625){}, quant_min=0, quant_max=255, qscheme=torch.per_tensor_affine, ch_axis=None, is_dynamic=False), weight=QuantizationSpec(dtype=torch.int8, observer_or_fake_quant_ctr=functools.partial(<class 'torchao.quantization.pt2e.observer.PerChannelMinMaxObserver'>, eps=0.000244140625){}, quant_min=-128, quant_max=127, qscheme=torc

# PTQ calibration

In [ ]:
from torch.utils.data import DataLoader, Subset
from torchao.quantization.pt2e import move_exported_model_to_eval

CALIBRATION_SIZE = 1000

calibration_indices = torch.randperm(
    len(train_dataset),
    generator=torch.Generator().manual_seed(SEED),
)[:CALIBRATION_SIZE].tolist()

calibration_dataset = Subset(
    train_dataset,
    calibration_indices,
)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=True,
)

# Exported models require the TorchAO helper
prepared_model = move_exported_model_to_eval(prepared_model)

# Calibrate using training images
with torch.inference_mode():
    for images, _ in calibration_loader:
        prepared_model(images.cpu())

print(f"Calibration completed on {len(calibration_dataset)} images.")

Calibration completed on 1000 images.


# Converting the model

In [ ]:
from torchao.quantization.pt2e import move_exported_model_to_eval

quantized_model = convert_pt2e(prepared_model)
quantized_model = move_exported_model_to_eval(quantized_model)

print("Quantization conversion completed.")

Quantization conversion completed.


# Compiling the quantized model

In [ ]:
quantized_model = torch.compile(
    quantized_model,
    backend="inductor",
)

print("Quantized model compiled.")

Quantized model compiled.


# Testing a batch to make sure compiled model runs

In [ ]:
images, labels = next(iter(test_loader))
images = images.cpu()

with torch.inference_mode():
    outputs = quantized_model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Output dtype:", outputs.dtype)
print("Test batch executed successfully.")

Input shape: torch.Size([25, 3, 256, 256])
Output shape: torch.Size([25, 10])
Output dtype: torch.float32
Test batch executed successfully.


# Evaluating quantized (INT8) accuracy
Note: Before we evaluate INT8 accuracy, we will first need to import results from FP32 baseline so that we can compare the results afterwards

In [ ]:
# Importing the FP32 baseline data from the json created in earlier notebook
import json

from google.colab import drive
drive.mount('/content/drive')

with open("/content/drive/My Drive/Colab Notebooks/fp32_baseline.json", "r") as f:
    baseline_data = json.load(f)

if isinstance(baseline_data, list):
    baseline_data = baseline_data[0]

baseline_accuracy = baseline_data["accuracy_percent"]



# Now we start to evaluate quantized accuracy

from torchao.quantization.pt2e import move_exported_model_to_eval

quantized_model = move_exported_model_to_eval(quantized_model)


correct = 0
total = 0

with torch.inference_mode():
    for images, labels in test_loader:
        images = images.cpu()
        labels = labels.cpu()

        outputs = quantized_model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

int8_accuracy = 100.0 * correct / total

print(f"INT8 PTQ accuracy: {int8_accuracy:.2f}%")
print(f"FP32 baseline accuracy: {baseline_accuracy:.2f}%")
print(
    f"Accuracy change: "
    f"{int8_accuracy - baseline_accuracy:+.2f} percentage points"
)

Mounted at /content/drive
FP32 baseline accuracy: 92.40%
INT8 PTQ accuracy: 92.40%
FP32 baseline accuracy: 92.40%
Accuracy change: +0.00 percentage points
